# Innovaccer Healthcare Patient Operations Analytics\nEnd-to-end portfolio workflow: SQL → Python → Power BI-ready outputs.\n\nThe source data are synthetic and intentionally non-uniform.

In [ ]:
from pathlib import Path\nimport sqlite3\nimport pandas as pd\nimport numpy as np\n

In [ ]:
ROOT = Path.cwd().parent if Path.cwd().name == '03_python' else Path.cwd()\nDB = ROOT / '01_data_raw' / 'innovaccer_healthcare.db'\nOUT = ROOT / '05_outputs'\nconn = sqlite3.connect(DB)\npatients = pd.read_sql_query('SELECT * FROM patients', conn)\nencounters = pd.read_sql_query('SELECT * FROM encounters', conn)\ndepartments = pd.read_sql_query('SELECT * FROM departments', conn)\ndiagnosis = pd.read_sql_query('SELECT * FROM diagnosis_catalog', conn)\nconn.close()\nprint(patients.shape, encounters.shape)

## 1. Data quality checks

In [ ]:
checks = {\n 'duplicate_patient_ids': int(patients.Patient_ID.duplicated().sum()),\n 'duplicate_encounter_ids': int(encounters.Encounter_ID.duplicated().sum()),\n 'missing_region': int((patients.Region.fillna('')=='').sum()),\n 'missing_marital_status': int((patients.Marital_Status.fillna('')=='').sum()),\n 'missing_provider_id': int((encounters.Provider_ID.fillna('')=='').sum()),\n 'negative_billed': int((encounters.Billed_Amount<0).sum()),\n 'paid_gt_billed': int((encounters.Paid_Amount>encounters.Billed_Amount).sum()),\n}\nchecks

## 2. Cleaning and enrichment

In [ ]:
encounters.Service_Date = pd.to_datetime(encounters.Service_Date)\npatients['Region'] = patients.Region.fillna('').replace('', 'Unknown')\npatients['Marital_Status'] = patients.Marital_Status.fillna('').replace('', 'Unknown')\nencounters['Provider_ID'] = encounters.Provider_ID.fillna('').replace('', 'Unknown')\ndf = encounters.merge(departments,on='Department_ID').merge(patients[['Patient_ID','Payer','Gender','Region']],on='Patient_ID').merge(diagnosis[['Diagnosis_Code','Diagnosis_Name']],left_on='Primary_Diagnosis_Code',right_on='Diagnosis_Code')\ndf['Is_Admission'] = df.Encounter_Type.isin(['Inpatient','Observation']).astype(int)\ndf['Month'] = df.Service_Date.dt.to_period('M').astype(str)\ndf.head()

## 3. KPI analysis

In [ ]:
kpis = {\n 'encounters': len(df),\n 'unique_patients': df.Patient_ID.nunique(),\n 'admissions': int(df.Is_Admission.sum()),\n 'avg_wait_minutes': round(df.Wait_Time_Minutes.mean(),1),\n 'readmissions_30d': int(df.Readmission_30d.sum()),\n 'total_billed': round(df.Billed_Amount.sum(),2),\n}\nkpis

## 4. Anomaly detection

In [ ]:
wait_p95 = df.Wait_Time_Minutes.quantile(.95)\nbill_p99 = df.Billed_Amount.quantile(.99)\ninpt_p99 = df.loc[df.Encounter_Type=='Inpatient','Length_of_Stay_Days'].quantile(.99)\nanomalies = df[(df.Wait_Time_Minutes>wait_p95) | (df.Billed_Amount>bill_p99) | ((df.Encounter_Type=='Inpatient') & (df.Length_of_Stay_Days>inpt_p99))]\nprint(wait_p95, bill_p99, inpt_p99, len(anomalies))

## 5. Export Power BI-ready tables

In [ ]:
OUT.mkdir(exist_ok=True)\ndf.to_csv(OUT/'fact_encounters_enriched.csv', index=False)\nprint('Saved Power BI-ready fact table.')